# 1일차 파트 2 실습 풀이

`1일차_02_실습.ipynb`의 풀이와 해설입니다. 프롬프트는 하나의 예시입니다. 모델의 응답은 실행할 때마다 다르므로 여러분의 프롬프트와 결과가 풀이와 달라도 테스트를 통과하면 됩니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 2 알람 로그를 분류하는 프롬프트 작성

**목적** 역할 지정, 맥락, 분류 기준, 출력 형식을 넣은 프롬프트로 분류 결과가 실행마다 같아지는지 확인합니다. 예시는 세 번의 분류가 달라질 때 더합니다.

**재료** `data/1일차/알람로그_10건.txt`(야간 설비 알람 로그 10건)와 `data/1일차/알람_분류기준.txt`(조치 필요로 분류하는 기준 세 가지)입니다. 줄별 정답은 `알람로그_정답.txt`에 있고, 테스트 셀이 읽습니다. 먼저 열어 보지 않고 풀어 봅니다.

**할 일**

1. 알람 로그와 분류 기준을 받아 프롬프트 문자열을 돌려주는 `build_classify_prompt()`를 작성합니다. 프롬프트에는 역할 지정, 맥락(누가 읽고 어디에 쓰는지), 분류 기준, 출력 형식이 들어가야 합니다. 예시(Few-shot)는 처음에는 넣지 않아도 되고, 4번에서 분류가 바뀐 줄이 있으면 더합니다.
2. 출력은 코드가 읽을 수 있게 한 줄에 하나씩 `번호 | 분류`로만 받습니다. 분류는 `조치 필요` 또는 `모니터링`입니다.
3. 테스트 셀을 실행합니다. 같은 프롬프트를 새 채팅에서 세 번 보내 응답을 검사합니다.
4. 테스트에서 실패한 항목이 있으면 프롬프트를 고쳐 다시 실행합니다. 분류가 바뀐 줄이 있으면 그 줄의 원문을 읽고, 분류 기준 어느 항목이 적용되는지 확인합니다.

**완료 기준** 테스트 셀이 "실습 2 통과"를 출력합니다. 세 번 모두 10건을 읽을 수 있고, 형식 외의 말이 없고, 세 번의 분류가 같고, 정답과 9건 이상 일치해야 합니다.

**코드에 나오는 이름** 테스트 셀의 `chat.say(프롬프트, show=False)`는 응답을 화면에 출력하지 않고 보냅니다. 받은 응답 텍스트는 `chat.last.text`로 읽습니다.

<details><summary>힌트 1</summary>

분류 기준 파일의 글을 프롬프트에 그대로 넣습니다. 기준을 말로 풀어 쓰면 모델이 기준을 다르게 읽을 수 있습니다.
</details>

<details><summary>힌트 2</summary>

세 번의 분류가 같지 않으면 분류가 바뀐 줄을 찾습니다. 그 줄과 비슷한 지어낸 줄을 Few-shot 예시로 한두 개 넣습니다. 로그의 줄을 그대로 예시로 넣으면 그 줄의 분류는 확인할 수 없으므로, 기준은 같고 내용이 다른 줄을 지어냅니다.
</details>

<details><summary>힌트 3</summary>

"다른 말, 설명, 코드 블록 표시는 쓰지 마"를 적지 않으면 응답 앞뒤에 인사말이나 표 머리글이 붙어 형식 검사에서 실패합니다.
</details>

In [ ]:
# 실습 2 재료 (그대로 실행합니다)
import checks_d1_2

log = read_material('알람로그_10건.txt')
rule = read_material('알람_분류기준.txt')
print(log)
print()
print(rule)

In [ ]:
# 실습 2 (1) 분류 프롬프트를 작성합니다
def build_classify_prompt(log, rule):
    """알람 로그와 분류 기준을 받아 모델에 보낼 프롬프트 문자열을 돌려줍니다."""
    return ('너는 반도체 장비팀 설비 엔지니어다. 이 자료는 아침 인수인계 때 주간조 파트장이 읽고 '
            '오늘 먼저 조치할 설비를 정하는 데 쓴다.\n'
            '다음 야간 설비 알람 로그를 [조치 필요 / 모니터링] 두 가지로 분류해줘.\n\n'
            + rule + '\n'
            '예시:\n'
            '- "9호기 칠러 유량 저하 경고, 원인 미상" → 조치 필요\n'
            '- "4호기 센서 오차 경고, 재시작 후 정상" → 모니터링\n\n'
            '출력은 한 줄에 하나씩 "번호 | 분류" 형식으로만 적고, 분류는 조치 필요 또는 모니터링 가운데 '
            '하나만 써. 다른 말, 설명, 코드 블록 표시는 쓰지 마.\n\n' + log)

In [ ]:
# 실습 2 (2) 같은 프롬프트를 새 채팅에서 세 번 보내고 테스트합니다 (실제 API를 부릅니다. 3회)
classify_prompt = build_classify_prompt(log, rule)
texts = []
for number in range(3):
    chat = llm.Chat(model='소형')
    chat.say(classify_prompt, show=False)
    texts.append(chat.last.text)
print(texts[0])
print()
test_results = checks_d1_2.check_classify(texts)

## 실습 3 분류 결과에서 표를 만들고 개수를 확인하는 체인

**목적** 실습 2의 분류 결과를 체인의 첫째 단계(1단)의 결과로 삼아 둘째 단계(2단, 마크다운 표 만들기)를 보내고, 표에서 센 행 수를 모델에게 묻지 않고 코드로 확인합니다. 제약 한 줄을 빼면 표가 어떻게 바뀌는지 비교합니다.

**재료** 실습 2의 `texts[0]`(1단의 분류 결과). 실습 2를 먼저 실행해야 합니다. 1단의 분류가 한 건이라도 틀리면 표를 바르게 만들어도 테스트 2~4번 가운데 하나 이상이 실패합니다. 실습 2의 분류가 정답과 10건 모두 일치한 결과로 진행합니다.

**표의 용도** 주간조 파트장이 아침 인수인계 때 이 표를 보고 먼저 조치할 설비를 정합니다.

**코드에 나오는 이름** `chat.history`는 그 채팅의 대화 이력(메시지 목록)입니다. 테스트 셀은 여기에 실습 2의 프롬프트와 응답을 직접 넣어, 1단을 다시 보내지 않고 2단을 이어서 보냅니다.

**할 일**

1. 마크다운 표의 데이터 줄 수를 세는 `count_table_rows()`를 작성합니다. 머리글 줄과 구분선(`| --- |`)은 세지 않고, 표 앞뒤의 문장도 세지 않습니다.
2. 2단 프롬프트 `STEP2_PROMPT`를 작성합니다. 조치 필요인 로그만 한 줄을 한 행으로 정리하고, 열은 `로그 번호 | 설비 | 알람 | 발생 시각`이며, 원자료에 없는 값은 `확인 필요`로 적게 합니다. 표 밖의 말은 쓰지 않게 합니다. `MERGE_BAN`의 문장을 그대로 프롬프트에 넣습니다.
3. 테스트 셀을 실행합니다. 1단 결과를 대화 이력에 넣고 2단을 보냅니다.
4. `MERGE_BAN`을 뺀 프롬프트를 두 번 보내 표에서 센 행 수와 로그 번호가 어떻게 바뀌는지 봅니다.
5. 표를 아래 루브릭 네 항목으로 직접 채점합니다. 항목마다 1~3점입니다. 이번에는 모델에게 채점을 맡기지 않습니다.
   - 빠짐없음: 1점 조치 필요 로그가 여럿 빠짐 / 2점 하나 빠짐 / 3점 모두 있음
   - 정확함: 1점 로그에 없는 값이 있음 / 2점 값이 다른 칸에 들어갔거나 로그로 채울 수 있는 칸이 비어 있음 / 3점 모든 칸이 로그와 일치
   - 바로 쓸 수 있음: 1점 다시 정리해야 함 / 2점 몇 칸을 고치면 됨 / 3점 그대로 인수인계에 씀
   - 목적 맞춤: 1점 어느 설비를 먼저 조치할지 알 수 없음 / 2점 일부 행만 조치 이유를 알 수 있음 / 3점 행마다 조치 이유(재발, 상한 근접, 원인 미상)를 알 수 있음

**완료 기준** 테스트 셀이 "실습 3 통과"를 출력합니다. 표에서 센 행 수와 로그 번호가 정답의 조치 필요 건과 같아야 합니다.

<details><summary>힌트 1</summary>

데이터 줄은 `|`로 시작하고 첫 칸이 숫자입니다. 정규식 `^\s*\|\s*\d+\s*\|`로 줄마다 확인하면 머리글과 구분선이 빠집니다.
</details>

<details><summary>힌트 2</summary>

같은 설비의 두 줄(5호기 E-201의 1번과 4번)이 한 행으로 합쳐지면 센 행 수가 정답보다 적어집니다. 합치지 않는다는 문장을 `MERGE_BAN`으로 넣는 이유가 이것입니다.
</details>

In [ ]:
# 실습 3 (1) 표의 데이터 줄 수를 세는 함수를 작성합니다
import re


def count_table_rows(text):
    """마크다운 표의 데이터 줄 수를 돌려줍니다. 머리글 줄, 구분선, 표 밖의 문장은 세지 않습니다."""
    return len([line for line in text.splitlines() if re.match(r'^\s*\|\s*\d+\s*\|', line)])

In [ ]:
# 실습 3 (2) 2단 프롬프트를 작성합니다
MERGE_BAN = '같은 설비가 두 번 나와도 행을 합치지 마.'          # 이 문장을 그대로 프롬프트에 넣습니다
STEP2_PROMPT = ('위 [조치 필요]에서 로그 한 줄을 표 한 행으로 정리해줘. ' + MERGE_BAN + '\n'
                '열: 로그 번호 | 설비 | 알람 | 발생 시각\n'
                "원자료에 없으면 '확인 필요'로 적어줘. 표 외의 다른 말은 하지 마.")

In [ ]:
# 실습 3 (3) 1단 결과를 대화 이력에 넣고 2단을 보내 테스트합니다 (실제 API를 부릅니다. 1회)
def start_chain():
    """실습 2의 프롬프트와 첫 응답을 이력으로 가진 채팅을 돌려줍니다."""
    chat = llm.Chat(model='소형')
    chat.history = [{'role': 'user', 'content': classify_prompt},
                    {'role': 'assistant', 'content': texts[0]}]
    return chat


chat = start_chain()
chat.say(STEP2_PROMPT)
print()
test_results = checks_d1_2.check_chain(chat.last.text, count_table_rows, texts[0])

In [ ]:
# 실습 3 (4) 합치지 말라는 문장을 빼고 두 번 보냅니다 (실제 API를 부릅니다. 2회)
for number in [1, 2]:
    chat = start_chain()
    chat.say(STEP2_PROMPT.replace(MERGE_BAN, ''), show=False)
    print(f'[{number}번째] 표의 행 수 {count_table_rows(chat.last.text)}, '
          f'로그 번호 {checks_d1_2.table_numbers(chat.last.text)}')

**해설**

아래는 2026-11-23에 Claude 모델로 실행한 결과입니다. 여러분의 응답은 다를 수 있습니다.

| 확인한 것 | 풀이 실행 결과 (소형 모델) |
| --- | --- |
| 실습 2 | 세 번 모두 같은 분류였고, 정답과 10건 모두 일치함. 테스트 4개 통과 |
| 실습 3 표 | 7행. 로그 번호 1, 2, 4, 5, 7, 8, 9로 정답의 조치 필요 건과 같음. 테스트 4개 통과 |
| 합치지 말라는 문장을 뺀 두 번 | 두 번 모두 7행이고 로그 번호도 같음. 이 모델에서는 문장이 없어도 같은 설비의 두 줄을 합치지 않았음 |

표의 내용을 읽으면 확인할 곳이 남아 있습니다. 로그 번호 4번의 행에서 알람 칸은 "확인 필요(원자료에 '재발'만 기재)"입니다. 1번 줄을 보면 E-201이 펌프 압력 이상임을 알 수 있는데도 알람 칸을 채우지 않았고, 알람 코드 E-201은 1번과 4번 행 모두 설비 칸에 들어갔습니다. 형식과 행 수는 테스트를 통과했지만, 파트장이 4번 행의 알람 칸만 보면 무슨 알람인지 알 수 없습니다.

루브릭으로 채점하면 다음과 같습니다.

- 빠짐없음 3점: 조치 필요 로그 7건이 모두 있습니다.
- 정확함 2점: 로그에 없는 값은 없지만, 알람 코드 E-201이 설비 칸에 들어갔고 4번 행의 알람 칸은 1번 줄로 채울 수 있는데 비어 있습니다.
- 바로 쓸 수 있음 2점: 4번 행의 알람 칸과 1번, 4번 행의 설비 칸을 고치면 인수인계에 쓸 수 있습니다.
- 목적 맞춤 2점: 1번(3회째), 5번(상한 근접), 7번(2회), 8번(원인 미상), 9번(재발)은 조치 이유가 보입니다. 2번은 "1회, 자동 복귀"로 적혀 있어, 9번에서 재발했다는 이유가 그 행에는 없습니다. 4번은 알람 칸이 비어 있습니다.

반대로 문장을 뺀 실험에서 행이 합쳐지지 않았다는 결과는 제약이 필요 없다는 뜻이 아닙니다. 모델과 응답에 따라 합쳐질 수 있으므로 제약은 남기고, 행 수는 코드로 확인합니다.